# CDS Pricing: Bootstrapping Default Intensities

*Nova SBE, Credit Risk (2026).*

Given a CDS spread curve for one reference entity, this notebook

1. bootstraps a piecewise-constant term structure of default intensities from the 1y, 3y and 5y quotes,
2. prices a 4-year CDS off the calibrated survival curve, and
3. marks to market a protection position that was bought one year ago at 60bp.

In [1]:
import numpy as np
import pandas as pd

from quantfin.cds import CDSMarket, bootstrap, fair_spread, mark_to_market, rpv01

## Inputs

Quoted spreads for 1, 3 and 5 years, quarterly premium payments, 40% recovery and a flat 3.5% risk-free rate (continuously compounded). The position is $5M notional of protection bought at 60bp.

In [2]:
market = CDSMarket(rate=0.035, recovery=0.40, period=0.25)

maturities = np.array([1.0, 3.0, 5.0])
quoted_spreads = np.array([70, 95, 120]) / 10_000

traded_spread = 60 / 10_000  # protection bought one year ago
notional = 5e6

display(pd.DataFrame({"Maturity": maturities, "Spread (bp)": quoted_spreads * 10_000}))

,Maturity,Spread (bp)
0,1.0,70.0
1,3.0,95.0
2,5.0,120.0


## Pricing

The pricing code lives in [`quantfin/cds.py`](../src/quantfin/cds.py). The default intensity $\lambda(t)$ is piecewise constant between the quoted maturities, so survival is

$$P_S(T) = \exp\left(-\int_0^T \lambda(s)\,ds\right).$$

- **Risky PV01:** discounted premium payments weighted by survival. Premium accrued since the last payment date is paid on default, approximated as half a period.
- **Default leg:** $(1-R)$ times the discounted probability of default in each payment period.
- **Fair spread:** default leg divided by risky PV01.
- **Mark to market** (protection buyer): $(s_t - s_0) \times \text{RPV01}_t \times N$.

## 1. Bootstrapping the default-intensity curve

Solve for one intensity at a time. $\lambda_1$ reprices the 1y quote alone; $\lambda_2$ then reprices the 3y quote with $\lambda_1$ fixed, and $\lambda_3$ the 5y quote with both fixed.

In [3]:
curve = bootstrap(maturities, quoted_spreads, market)

intensities = pd.DataFrame(
    {"Lambda": curve.intensities, "Lambda (bp)": np.array(curve.intensities) * 10_000},
    index=["0-1y", "1-3y", "3-5y"],
)
display(intensities.round(4))

,Lambda,Lambda (bp)
0-1y,0.0117,116.6667
1-3y,0.0181,180.8128
3-5y,0.0272,271.9720


## 2. Fair spread of a 4-year CDS

The 4-year maturity is not quoted, so its spread follows from the calibrated curve. The 3–4y period uses $\lambda_3$.

In [4]:
s_t = fair_spread(curve, market, maturity=4.0)

print(f"4-year fair CDS spread: {s_t * 10_000:.4f} bp")

4-year fair CDS spread: 110.6726 bp


## 3. Mark-to-market of the 60bp protection position

One year after inception the protection has four years left, so it is valued against the 4-year fair spread from step 2. Spreads have widened since purchase, so the protection buyer is in the money.

In [5]:
annuity = rpv01(curve, market, maturity=4.0)
mtm = mark_to_market(s_t, traded_spread, annuity, notional)

position = pd.DataFrame(
    {
        "s_0 (bp)": traded_spread * 10_000,
        "s_t (bp)": s_t * 10_000,
        "RPV01_t": annuity,
        "MtM ($)": mtm,
    },
    index=["Values"],
).T
display(position.round(4))

,Values
s_0 (bp),60.0000
s_t (bp),110.6726
RPV01_t,3.6043
MtM ($),91319.9526
